In [5]:
!pip install -U \
langchain==0.3.27 \
langchain-core==0.3.79 \
langchain-community==0.3.31 \
langchain-openai==0.3.35 \
langchain-text-splitters==0.3.9 \
langchainhub \
faiss-cpu \
sentence-transformers -q

In [ ]:
import os
from langchain.document_loaders import PyPDFLoader # a class used to load text from PDF files

# Imports a powerful text-splitting tool that breaks large documents into manageable chunks for embedding.
# LLMs have context length limits, so splitting documents into chunks is essential.
# Why recursive?
# It tries to split at logical boundaries (like paragraphs, sentences, etc.) before falling back to raw character limits.
from langchain.text_splitter import RecursiveCharacterTextSplitter

# OpenAIEmbeddings - to convert text into vector format using OpenAI’s embedding API
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

from langchain.vectorstores import FAISS # a fast in-memory vector search engine used for semantic search.
from langchain.chains import RetrievalQA

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.1.1
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [7]:
!pip install pypdf -q

In [ ]:
path = './the_nestle_hr_policy_pdf_2012.pdf'

loader = PyPDFLoader(path)
documents = loader.load()
# Print number of pages loaded
print(f"Loaded {len(documents)} pages from the PDF.")

Loaded 8 pages from the PDF.


In [9]:
# https://miro.medium.com/v2/resize:fit:1400/1*yfeUrFCr9oEVZofS8TvDEg.png
# https://tiktokenizer.vercel.app/
# RecursiveCharacterTextSplitter tries to split at logical boundaries (paragraph → sentence → word → character) — hence, recursive
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, # chunk_size=500: Each chunk will be up to 500 characters long.
    chunk_overlap=100 # chunk_overlap=100: The last 100 characters of one chunk are repeated in the next chunk for context continuity.
)
chunks = text_splitter.split_documents(documents)

# Print number of chunks created
print(f"Splitted into {len(chunks)} chunks.")
print(chunks[0])

Splitted into 39 chunks.
page_content='Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy' metadata={'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2013-02-12T08:06:14+01:00', 'moddate': '2026-04-23T10:23:08-07:00', 'trapped': '/False', 'source': './docs/the_nestle_hr_policy_pdf_2012.pdf', 'total_pages': 8, 'page': 0, 'page_label': '1'}


In [10]:
embeddings = OpenAIEmbeddings() # text-embedding-ada-002


In [11]:
!pip install faiss-cpu -q

In [12]:
from langchain.vectorstores import FAISS
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
vectorstore = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
print("Vector store loaded from disk successfully.")
print(f"Number of vectors in the vector store: {vectorstore.index.ntotal}")

Vector store loaded from disk successfully.
Number of vectors in the vector store: 39


In [13]:
llm = ChatOpenAI(
    model="gpt-4-turbo", # Use the GPT-4 Turbo model for faster and more cost-effective responses
)

In [14]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

rag_chain = RetrievalQA.from_chain_type(
    llm=llm, # Use the initialized LLM (GPT-4 Turbo)
    retriever=retriever, # Use the retriever to fetch relevant chunks
    return_source_documents=True # Return the source documents along with the answer
)

In [15]:
def ask_hr_agent(query: str):
    print(f"\nUser Query: {query}")
    result = rag_chain.invoke({"query": query})
    print()
    print("\nAnswer:")
    print(result["result"])
    print("\nRetrieved Passages:")
    for doc in result["source_documents"]:
        print("-->", doc.page_content.strip()[:200], "...")
    return(result["result"])

In [16]:
ask_hr_agent("How does an employee get promoted?")


User Query: How does an employee get promoted?


Answer:
The process for employee promotion typically involves several key elements as outlined in the information provided:

1. **Performance Evaluation Process (PE)**: Employees are regularly assessed on their performance through the Performance Evaluation process. Their achievements, contributions, and overall effectiveness in their current role are evaluated.

2. **Progress and Development Guide (PDG)**: This tool seems to focus on the personal and professional development of employees, likely identifying areas for growth and skills enhancement that align with career aspirations and business needs.

3. **360° Assessments**: These assessments provide comprehensive feedback from various sources, such as peers, supervisors, and possibly subordinates. This feedback can be instrumental in highlighting both strengths and areas for improvement, impacting decisions about promotions.

4. **Regular Coaching and Monitoring**: Managers take an a

'The process for employee promotion typically involves several key elements as outlined in the information provided:\n\n1. **Performance Evaluation Process (PE)**: Employees are regularly assessed on their performance through the Performance Evaluation process. Their achievements, contributions, and overall effectiveness in their current role are evaluated.\n\n2. **Progress and Development Guide (PDG)**: This tool seems to focus on the personal and professional development of employees, likely identifying areas for growth and skills enhancement that align with career aspirations and business needs.\n\n3. **360° Assessments**: These assessments provide comprehensive feedback from various sources, such as peers, supervisors, and possibly subordinates. This feedback can be instrumental in highlighting both strengths and areas for improvement, impacting decisions about promotions.\n\n4. **Regular Coaching and Monitoring**: Managers take an active role in monitoring objectives and providing

In [17]:
%pip install gradio --upgrade

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [20]:
import gradio as gr

# Step 4: Create Gradio Interface
gradio_interface = gr.Interface(
    fn=ask_hr_agent,                           # link our function
    inputs=gr.Textbox(label="Enter Your Question"),
    outputs=gr.Textbox(label="Answer"),
    title="Nestle AI HR Assisstant",
    description="Ask any question about Nestle's Human Resources Policy"
)

# Step 5: Launch the app in notebook mode
gradio_interface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://47620dd29b01d21edc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



User Query: How does an employee get promoted?


Answer:
Employees at the company are typically promoted based on their performance evaluations, development through the Progress and Development Guide (PDG), and feedback from 360° assessments. Managers are responsible for monitoring objectives and providing regular coaching throughout the year. Promotion decisions also consider an employee's career aspirations and are supported by discussions with line managers. This holistic approach ensures that promotions are aligned with both individual performance and professional development goals.

Retrieved Passages:
--> managed with integrity.
Employees receive regular feedback on their 
performance and career aspirations through a variety of tools and processes such as the 
Performance Evaluation process (PE), the   ...
--> management maintains with all employees.
In the spirit of continuous improvement, we 
encourage two-way dialogue with our employees that goes beyond the traditional aspect